# LinkedIn Job Market Analysis & Career Insights

**An end-to-end reproducible data-analysis workflow**

Dataset: `linkdin_Job_data_cleaned.csv` — 5,819 job postings × 16 columns  
Source: Real LinkedIn job-posting data (India-centric, technology-sector focus)

---

## Table of Contents
1. [Import Libraries](#1)
2. [Load the Dataset](#2)
3. [Data Quality Inspection](#3)
4. [Data Cleaning & Preprocessing](#4)
5. [Exploratory Data Analysis](#5)
6. [Statistical Summaries](#6)
7. [Visualizations](#7)
8. [Key Findings & Observations](#8)
9. [Machine Learning Workflow](#9)
10. [Target Definition & Leakage Prevention](#10)
11. [Model Training & Evaluation](#11)
12. [Accuracy, Precision, Recall & ROC-AUC](#12)
13. [Business Insights & Recommendations](#13)

---
<a id='1'></a>
## 1. Import Libraries

In [ ]:
import re
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, confusion_matrix, precision_score,
    recall_score, roc_auc_score, roc_curve, ConfusionMatrixDisplay
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# ── Plot style ────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (10, 5)})

# ── Reproducibility ───────────────────────────────────────────
RANDOM_STATE = 42

print('Libraries loaded successfully.')
print(f'pandas {pd.__version__}  |  numpy {np.__version__}')

---
<a id='2'></a>
## 2. Load the LinkedIn Job Dataset

In [ ]:
# ── Load the cleaned dataset ──────────────────────────────────
CSV_PATH = 'linkdin_Job_data_cleaned.csv'

df = pd.read_csv(CSV_PATH)

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Columns: {df.columns.tolist()}')

In [ ]:
# First 5 rows
df.head()

In [ ]:
# Column data types
df.dtypes

---
<a id='3'></a>
## 3. Data Quality Inspection

> The raw data (`linkdin_Job_data.csv`) went through a multi-step cleaning pipeline
> (`clean_data.py`) before this notebook. This section audits the **cleaned** dataset.

**Issues found and fixed in the cleaning pipeline:**
- `Column1` and `company_id` were 100 % null → dropped
- Fully duplicate rows removed
- Skeleton rows (all key fields NaN) removed
- `no_of_application`: time-unit-only values (`'hours'`, `'days'`) → set to NaN; remainder cast to int
- Same `job_ID` duplicates (scraper re-collected): kept row with highest applicant count
- `no_of_employ` split into `emp_size` + `emp_industry`
- `full_time_remote` split into `employment_type` + `seniority_level`
- `linkedin_followers`: city/county-name values → NaN; remaining extracted as integers
- `alumni` column: `"N company alumni"` → integer `alumni_count`

In [ ]:
# ── Missing-value audit ───────────────────────────────────────
missing = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %':     (df.isnull().sum() / len(df) * 100).round(1)
}).sort_values('Missing %', ascending=False)

print('Missing values per column:')
print(missing[missing['Missing Count'] > 0].to_string())

In [ ]:
# ── Visualise missing data ────────────────────────────────────
miss_plot = missing[missing['Missing Count'] > 0].reset_index()
miss_plot.columns = ['Column', 'Missing Count', 'Missing %']

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(miss_plot['Column'], miss_plot['Missing %'],
               color='#3b82d4', edgecolor='white')
for bar, pct in zip(bars, miss_plot['Missing %']):
    ax.text(bar.get_width() + 0.4, bar.get_y() + bar.get_height() / 2,
            f'{pct:.1f}%', va='center', fontsize=9)
ax.set_xlabel('Missing (%)')
ax.set_title('Missing Values by Column (cleaned dataset)', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# ── Duplicate check ───────────────────────────────────────────
dup_rows  = df.duplicated().sum()
dup_ids   = df['job_ID'].duplicated().sum()

print(f'Fully duplicate rows : {dup_rows}')
print(f'Duplicate job_IDs    : {dup_ids}')
assert dup_rows == 0, 'Unexpected duplicate rows found!'
assert dup_ids  == 0, 'Unexpected duplicate job_IDs found!'
print('Assertion passed: no duplicates in cleaned dataset.')

In [ ]:
# ── Cardinality of categorical columns ───────────────────────
cat_cols = ['work_type', 'employment_type', 'seniority_level', 'emp_size', 'emp_industry']
for col in cat_cols:
    n_unique = df[col].nunique()
    top_val  = df[col].value_counts().index[0] if df[col].notna().any() else 'N/A'
    top_pct  = df[col].value_counts().iloc[0] / df[col].notna().sum() * 100
    print(f'{col:<20} unique={n_unique:>4}   top="{top_val}" ({top_pct:.1f}%)')

---
<a id='4'></a>
## 4. Data Cleaning & Preprocessing

The cleaning was performed in `clean_data.py`. Here we **reproduce** the key
derived columns needed for analysis and ML.

In [ ]:
# ── Derive: hours_since_posted ────────────────────────────────
def _to_hours(val) -> float:
    """Convert posted_day_ago string (e.g. '3 days', '10 hours') to numeric hours."""
    if pd.isna(val):
        return np.nan
    v = str(val).strip().lower()
    for pat, mult in [
        (r'(\d+)\s*second', 1 / 3600),
        (r'(\d+)\s*minute', 1 / 60),
        (r'(\d+)\s*hour',   1),
        (r'(\d+)\s*day',    24),
        (r'(\d+)\s*week',   168),
    ]:
        m = re.match(pat, v)
        if m:
            return float(m.group(1)) * mult
    return np.nan

df['hours_since_posted'] = df['posted_day_ago'].apply(_to_hours)

# ── Derive: city, country ─────────────────────────────────────
df['city']    = df['location'].apply(
    lambda loc: str(loc).split(',')[0].strip() if pd.notna(loc) else np.nan
)
df['country'] = df['location'].apply(
    lambda loc: str(loc).split(',')[-1].strip() if pd.notna(loc) else np.nan
)

# ── Derive: is_remote (binary) ────────────────────────────────
df['is_remote'] = (df['work_type'].str.strip().str.lower() == 'remote').astype(int)

# ── Derive: log_followers (log1p-scaled) ─────────────────────
df['log_followers'] = np.log1p(df['linkedin_followers'].fillna(0))

print('Derived columns created:')
for c in ['hours_since_posted', 'city', 'country', 'is_remote', 'log_followers']:
    print(f'  {c}: non-null={df[c].notna().sum():,}  sample={df[c].dropna().iloc[0]}')

In [ ]:
# ── Company-size ordered category ────────────────────────────
SIZE_ORDER = [
    '1-10 employees', '11-50 employees', '51-200 employees',
    '201-500 employees', '501-1,000 employees', '1,001-5,000 employees',
    '5,001-10,000 employees', '10,001+ employees',
]
df['emp_size'] = pd.Categorical(df['emp_size'], categories=SIZE_ORDER, ordered=True)

print('emp_size value counts (ordered):')
print(df['emp_size'].value_counts().reindex(SIZE_ORDER))

---
<a id='5'></a>
## 5. Exploratory Data Analysis

EDA is organized across five dimensions:
1. **Geographic distribution** — where are jobs posted?
2. **Job roles & demand** — which titles are most-posted?
3. **Employment characteristics** — type, seniority, work arrangement
4. **Company profiles** — size, industry, LinkedIn presence
5. **Applicant behaviour** — volume, timing, competition

In [ ]:
# ── 5.1 Geographic distribution ──────────────────────────────
top_locations = df['location'].value_counts().head(12)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(top_locations.index[::-1], top_locations.values[::-1],
               color='#3b82d4', edgecolor='white')
for bar, val in zip(bars, top_locations.values[::-1]):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height() / 2,
            f'{val:,}', va='center', fontsize=9)
ax.set_xlabel('Number of Job Postings')
ax.set_title('Top 12 Locations by Job Postings', fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nTotal unique locations: {df["location"].nunique():,}')
print(f'Top location: {top_locations.index[0]} ({top_locations.iloc[0]:,} postings)')

In [ ]:
# ── 5.2 Top job titles ────────────────────────────────────────
top_jobs = df['job'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top_jobs.index[::-1], top_jobs.values[::-1],
               color='#7c5cd8', edgecolor='white')
for bar, val in zip(bars, top_jobs.values[::-1]):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
            str(val), va='center', fontsize=9)
ax.set_xlabel('Number of Postings')
ax.set_title('Top 15 Most-Posted Job Titles', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.3 Employment type distribution ─────────────────────────
emp_vc = df['employment_type'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie chart
colors = ['#3b82d4', '#f59e0b', '#7c5cd8', '#10b981', '#ef4444', '#6b7280', '#0ea5e9']
wedges, texts, autotexts = axes[0].pie(
    emp_vc.values, labels=emp_vc.index, autopct='%1.1f%%',
    colors=colors[:len(emp_vc)], startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=1.5)
)
axes[0].set_title('Employment Type Distribution', fontweight='bold')

# Bar chart
axes[1].bar(emp_vc.index, emp_vc.values, color=colors[:len(emp_vc)], edgecolor='white')
for i, (_, val) in enumerate(emp_vc.items()):
    axes[1].text(i, val + 20, str(val), ha='center', fontsize=9)
axes[1].set_ylabel('Count')
axes[1].set_title('Employment Type Count', fontweight='bold')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

ft_pct = emp_vc.get('Full-time', 0) / emp_vc.sum() * 100
print(f'Full-time share: {ft_pct:.1f}%  ({emp_vc.get("Full-time", 0):,} postings)')

In [ ]:
# ── 5.4 Seniority level distribution ─────────────────────────
sen_vc = df['seniority_level'].dropna().value_counts()

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(sen_vc.index, sen_vc.values,
              color=['#f59e0b', '#3b82d4', '#10b981', '#ef4444', '#7c5cd8', '#6b7280'][:len(sen_vc)],
              edgecolor='white')
for bar, val in zip(bars, sen_vc.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 15,
            str(val), ha='center', fontsize=9)
ax.set_ylabel('Number of Postings')
ax.set_title('Job Postings by Seniority Level', fontweight='bold')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

ms_pct = sen_vc.get('Mid-Senior level', 0) / sen_vc.sum() * 100
print(f'Mid-Senior level: {sen_vc.get("Mid-Senior level", 0):,} postings ({ms_pct:.1f}% of seniority-tagged)')
print(f'Note: {df["seniority_level"].isna().sum():,} postings ({df["seniority_level"].isna().mean()*100:.1f}%) have no seniority tag')

In [ ]:
# ── 5.5 Work type (Remote / On-site / Hybrid) ─────────────────
wt_vc = df['work_type'].dropna().value_counts()

fig, ax = plt.subplots(figsize=(6, 4))
color_map = {'Remote': '#3b82d4', 'On-site': '#f59e0b', 'Hybrid': '#7c5cd8'}
bar_colors = [color_map.get(k, '#aaa') for k in wt_vc.index]
bars = ax.bar(wt_vc.index, wt_vc.values, color=bar_colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, wt_vc.values):
    pct = val / wt_vc.sum() * 100
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 20,
            f'{val:,}\n({pct:.1f}%)', ha='center', fontsize=9)
ax.set_ylabel('Number of Postings')
ax.set_title('Work Type Distribution', fontweight='bold')
ax.set_ylim(0, wt_vc.max() * 1.2)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.6 Company size distribution ────────────────────────────
sz_vc = df['emp_size'].dropna().value_counts().reindex(SIZE_ORDER).dropna()

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(sz_vc.index, sz_vc.values, color='#10b981', edgecolor='white')
for bar, val in zip(bars, sz_vc.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 15,
            str(val), ha='center', fontsize=9)
ax.set_ylabel('Number of Postings')
ax.set_title('Job Postings by Company Size', fontweight='bold')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.7 Top industries ────────────────────────────────────────
ind_vc = df['emp_industry'].dropna().value_counts().head(10)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(ind_vc.index[::-1], ind_vc.values[::-1], color='#0ea5e9', edgecolor='white')
for bar, val in zip(bars, ind_vc.values[::-1]):
    ax.text(bar.get_width() + 10, bar.get_y() + bar.get_height() / 2,
            str(val), va='center', fontsize=9)
ax.set_xlabel('Number of Postings')
ax.set_title('Top 10 Industries by Job Postings', fontweight='bold')
plt.tight_layout()
plt.show()

it_pct = ind_vc.iloc[0] / df['emp_industry'].notna().sum() * 100
print(f'IT Services & Consulting: {ind_vc.iloc[0]:,} postings ({it_pct:.1f}% of industry-tagged)')

In [ ]:
# ── 5.8 Top companies by posting volume ──────────────────────
co_vc = df['company_name'].dropna().value_counts().head(12)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(co_vc.index[::-1], co_vc.values[::-1], color='#ef4444', edgecolor='white')
for bar, val in zip(bars, co_vc.values[::-1]):
    pct = val / len(df) * 100
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height() / 2,
            f'{val:,} ({pct:.1f}%)', va='center', fontsize=9)
ax.set_xlabel('Job Postings')
ax.set_title('Top 12 Companies by Job Postings', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.9 Applicant count distribution ─────────────────────────
app_df = df[df['no_of_application'].notna()].copy()
print(f'Postings with applicant data: {len(app_df):,} ({len(app_df)/len(df)*100:.1f}% of total)')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(app_df['no_of_application'], bins=40, color='#7c5cd8', edgecolor='white')
axes[0].axvline(app_df['no_of_application'].median(), color='#ef4444',
                linestyle='--', linewidth=1.8, label=f'Median={app_df["no_of_application"].median():.0f}')
axes[0].axvline(app_df['no_of_application'].mean(), color='#f59e0b',
                linestyle='--', linewidth=1.8, label=f'Mean={app_df["no_of_application"].mean():.1f}')
axes[0].set_xlabel('Number of Applicants')
axes[0].set_ylabel('Postings')
axes[0].set_title('Distribution of Applicants per Posting', fontweight='bold')
axes[0].legend()

# Box plot by work type
wt_order = ['Remote', 'On-site', 'Hybrid']
wt_data  = [app_df[app_df['work_type'] == w]['no_of_application'].dropna().values
             for w in wt_order if w in app_df['work_type'].values]
wt_labels = [w for w in wt_order if w in app_df['work_type'].values]
bp = axes[1].boxplot(wt_data, labels=wt_labels, patch_artist=True, notch=False)
box_colors = ['#3b82d4', '#f59e0b', '#7c5cd8']
for patch, color in zip(bp['boxes'], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_ylabel('Number of Applicants')
axes[1].set_title('Applicant Count by Work Type', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ── 5.10 Posting age vs applicant count scatter ───────────────
scatter_df = app_df.dropna(subset=['hours_since_posted', 'work_type']).copy()

fig, ax = plt.subplots(figsize=(10, 5))
color_map = {'Remote': '#3b82d4', 'On-site': '#f59e0b', 'Hybrid': '#7c5cd8'}
for wt, grp in scatter_df.groupby('work_type'):
    ax.scatter(grp['hours_since_posted'], grp['no_of_application'],
               label=wt, alpha=0.35, s=18, color=color_map.get(wt, '#aaa'))

# OLS trendline (pooled)
x = scatter_df['hours_since_posted'].values
y = scatter_df['no_of_application'].values
m, b = np.polyfit(x, y, 1)
x_line = np.linspace(x.min(), x.max(), 200)
ax.plot(x_line, m * x_line + b, color='#ef4444', linewidth=2,
        label=f'OLS trendline (slope={m:.2f})')

ax.set_xlabel('Hours Since Posted')
ax.set_ylabel('Number of Applicants')
ax.set_title('Posting Age vs Number of Applicants', fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

corr = scatter_df[['hours_since_posted', 'no_of_application']].corr().iloc[0, 1]
print(f'Pearson correlation (hours_since_posted vs applicants): {corr:.3f}')

In [ ]:
# ── 5.11 Average applicants by seniority ─────────────────────
sen_app = app_df[app_df['seniority_level'].notna()].groupby('seniority_level', observed=True)
avg_sen = (
    sen_app['no_of_application']
    .agg(mean='mean', count='count')
    .reset_index()
    .rename(columns={'mean': 'Avg Applicants', 'count': 'Postings'})
)
avg_sen = avg_sen[avg_sen['Postings'] >= 5].sort_values('Avg Applicants', ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(avg_sen['seniority_level'], avg_sen['Avg Applicants'],
              color='#f59e0b', edgecolor='white')
for bar, val in zip(bars, avg_sen['Avg Applicants']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f'{val:.1f}', ha='center', fontsize=9)
ax.set_ylabel('Average Applicants per Posting')
ax.set_title('Average Applicants by Seniority Level', fontweight='bold')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.12 LinkedIn followers distribution ─────────────────────
foll_df = df[df['linkedin_followers'].notna()]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Log-scale histogram
axes[0].hist(np.log10(foll_df['linkedin_followers'].clip(lower=1)),
             bins=40, color='#3b82d4', edgecolor='white')
axes[0].set_xlabel('Log10(LinkedIn Followers)')
axes[0].set_ylabel('Number of Postings')
axes[0].set_title('LinkedIn Followers Distribution (log10 scale)', fontweight='bold')

# Top 10 companies by avg followers
top_foll = (
    df.groupby('company_name')['linkedin_followers'].mean()
    .nlargest(10).reset_index()
)
top_foll['label'] = top_foll['linkedin_followers'].apply(
    lambda x: f'{x/1e6:.1f}M' if x >= 1e6 else f'{x/1e3:.0f}K'
)
axes[1].barh(top_foll['company_name'][::-1], top_foll['linkedin_followers'][::-1],
             color='#3b82d4', edgecolor='white')
for i, (_, row) in enumerate(top_foll[::-1].iterrows()):
    axes[1].text(row['linkedin_followers'] * 1.02, i, row['label'], va='center', fontsize=9)
axes[1].set_xlabel('Avg LinkedIn Followers')
axes[1].set_title('Top 10 Companies by LinkedIn Followers', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ── 5.13 Work type x Employment type heatmap ─────────────────
cross = pd.crosstab(df['work_type'], df['employment_type'])

fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(cross, annot=True, fmt='d', cmap='Blues', linewidths=0.5, ax=ax)
ax.set_title('Work Type × Employment Type — Posting Count', fontweight='bold')
ax.set_xlabel('Employment Type')
ax.set_ylabel('Work Type')
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.14 Avg applicants by company size ──────────────────────
app_sz = app_df[app_df['emp_size'].notna()].copy()
avg_sz = (
    app_sz.groupby('emp_size', observed=True)['no_of_application']
    .mean().reindex(SIZE_ORDER).dropna().round(1)
)

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(avg_sz.index, avg_sz.values, color='#10b981', edgecolor='white')
for bar, val in zip(bars, avg_sz.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f'{val:.1f}', ha='center', fontsize=9)
ax.set_ylabel('Avg Applicants per Posting')
ax.set_title('Average Applicants per Posting by Company Size', fontweight='bold')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

---
<a id='6'></a>
## 6. Statistical Summaries

In [ ]:
# ── Numeric column summary ────────────────────────────────────
num_cols = ['no_of_application', 'linkedin_followers', 'alumni_count', 'hours_since_posted']
summary = df[num_cols].describe().T
summary.insert(0, 'non_null', df[num_cols].notna().sum())
summary.insert(1, 'null_pct', (df[num_cols].isna().sum() / len(df) * 100).round(1))
summary = summary.round(2)
print('Numeric Column Statistical Summary:')
print(summary.to_string())

In [ ]:
# ── Applicant count detailed stats ───────────────────────────
app_stats = app_df['no_of_application'].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95])
print('Applicant Count — Detailed Statistics')
print('=' * 40)
for stat, val in app_stats.items():
    print(f'  {stat:<12}: {val:>8.1f}')
print(f'  {"Skewness":<12}: {app_df["no_of_application"].skew():>8.2f}')
print(f'  {"Kurtosis":<12}: {app_df["no_of_application"].kurt():>8.2f}')
print()
print('NOTE: LinkedIn caps displayed applicant count at 200.')
print(f'  Postings at cap (200): {(app_df["no_of_application"]==200).sum():,}')

In [ ]:
# ── Correlation matrix (numeric features) ────────────────────
corr_cols = ['no_of_application', 'hours_since_posted', 'log_followers',
             'alumni_count', 'is_remote']
corr_df = df[corr_cols].dropna()
corr_matrix = corr_df.corr()

fig, ax = plt.subplots(figsize=(7, 5))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax, mask=False)
ax.set_title('Pearson Correlation Matrix (numeric features)', fontweight='bold')
plt.tight_layout()
plt.show()

print('\nCorrelation with no_of_application:')
print(corr_matrix['no_of_application'].drop('no_of_application').sort_values(ascending=False).to_string())

In [ ]:
# ── Posting freshness summary ─────────────────────────────────
print('Posted Day Ago — Top 15 Values:')
print(df['posted_day_ago'].value_counts().head(15).to_string())
print()
print('Hours Since Posted — Summary:')
print(df['hours_since_posted'].describe().round(1).to_string())

---
<a id='7'></a>
## 7. All Important Visualizations

In [ ]:
# ── 7.1 Remote postings — top locations ──────────────────────
remote_df = df[df['work_type'] == 'Remote']
rem_loc = remote_df['location'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(rem_loc.index[::-1], rem_loc.values[::-1], color='#3b82d4', edgecolor='white')
for bar, val in zip(bars, rem_loc.values[::-1]):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height() / 2,
            str(val), va='center', fontsize=9)
ax.set_xlabel('Remote Postings')
ax.set_title('Top 10 Locations for Remote Postings', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 7.2 Applicant count percentile chart ─────────────────────
percentiles = np.arange(0, 101, 5)
pct_values  = np.percentile(app_df['no_of_application'].dropna(), percentiles)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(percentiles, pct_values, color='#7c5cd8', linewidth=2, marker='o', markersize=4)
ax.fill_between(percentiles, pct_values, alpha=0.15, color='#7c5cd8')
ax.axhline(app_df['no_of_application'].median(), color='#ef4444',
           linestyle='--', linewidth=1.5, label=f'Median = {app_df["no_of_application"].median():.0f}')
ax.set_xlabel('Percentile')
ax.set_ylabel('Applicant Count')
ax.set_title('Applicant Count — Cumulative Distribution', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 7.3 Posting volume by day-ago bucket ─────────────────────
def age_bucket(val):
    if pd.isna(val): return 'Unknown'
    if val <= 1:     return '<=1 hour'
    if val <= 12:    return '2-12 hours'
    if val <= 24:    return '13-24 hours'
    if val <= 72:    return '2-3 days'
    if val <= 168:   return '4-7 days'
    return '>7 days'

bucket_order = ['<=1 hour', '2-12 hours', '13-24 hours', '2-3 days', '4-7 days', '>7 days', 'Unknown']
df['age_bucket'] = df['hours_since_posted'].apply(age_bucket)
bucket_vc = df['age_bucket'].value_counts().reindex(bucket_order).dropna()

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(bucket_vc.index, bucket_vc.values, color='#0ea5e9', edgecolor='white')
for bar, val in zip(bars, bucket_vc.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
            str(val), ha='center', fontsize=9)
ax.set_ylabel('Number of Postings')
ax.set_title('Postings by Age at Time of Collection', fontweight='bold')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
# ── 7.4 Seniority x Work type stacked bar ────────────────────
sen_wt = (
    df[df['seniority_level'].notna() & df['work_type'].notna()]
    .groupby(['seniority_level', 'work_type'], observed=True)
    .size().unstack(fill_value=0)
)
# Keep only columns present
wt_present = [w for w in ['Remote', 'On-site', 'Hybrid'] if w in sen_wt.columns]
sen_wt = sen_wt[wt_present]

fig, ax = plt.subplots(figsize=(10, 5))
bottom = np.zeros(len(sen_wt))
colors = {'Remote': '#3b82d4', 'On-site': '#f59e0b', 'Hybrid': '#7c5cd8'}
for wt in wt_present:
    ax.bar(sen_wt.index, sen_wt[wt], bottom=bottom, label=wt,
           color=colors[wt], edgecolor='white')
    bottom += sen_wt[wt].values
ax.set_ylabel('Number of Postings')
ax.set_title('Seniority Level × Work Type', fontweight='bold')
ax.legend(title='Work Type')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

---
<a id='8'></a>
## 8. Key Findings & Observations

In [ ]:
# ── Print key metrics to support findings ────────────────────
total      = len(df)
companies  = df['company_name'].nunique()
remote_pct = df['is_remote'].mean() * 100
ft_pct     = (df['employment_type'] == 'Full-time').sum() / df['employment_type'].notna().sum() * 100
ms_pct     = (df['seniority_level'] == 'Mid-Senior level').sum() / df['seniority_level'].notna().sum() * 100
epam_pct   = df['company_name'].value_counts().iloc[0] / total * 100
it_pct2    = ind_vc.iloc[0] / df['emp_industry'].notna().sum() * 100

print('=== KEY FINDINGS =================================')
print(f'1. Dataset covers {total:,} postings from {companies:,} companies.')
print(f'2. Geographic concentration: Bengaluru is #1 ({top_locations.iloc[0]:,} postings).')
print(f'   Top 4 cities (Bengaluru, Hyderabad, Gurugram, Mumbai) dominate.')
print(f'3. Work type: Remote={remote_pct:.1f}% of postings — nearly equal to On-site.')
print(f'4. Employment type: Full-time dominates at {ft_pct:.1f}% of typed postings.')
print(f'5. Seniority: Mid-Senior level accounts for {ms_pct:.1f}% of tagged postings.')
print(f'   Entry-level roles are scarce (129 postings vs 2,965 Mid-Senior).')
print(f'6. Sector skew: IT Services & Consulting = {it_pct2:.1f}% of industry-tagged postings.')
print(f'7. Outlier: EPAM Anywhere = {epam_pct:.1f}% of ALL postings (single company).')
print(f'8. Top job titles: Lead Java Software Engineer ({top_jobs.iloc[0]:,}) &')
print(f'   Senior Automation Tester ({top_jobs.iloc[1]:,}).')
print(f'9. Applicant distribution: median={app_df["no_of_application"].median():.0f}, '
      f'mean={app_df["no_of_application"].mean():.1f}, max=200 (LinkedIn cap).')
print(f'   Right-skewed: 50% of postings get <=23 applicants.')
print(f'10. Posting age is the #1 predictor of applicant volume (more time = more apps).')
print('==================================================')

### Summary of Key Findings

| # | Finding | Implication |
|---|---------|-------------|
| 1 | 5,819 postings from 1,979 companies | Diverse employer base with heavy IT focus |
| 2 | Bengaluru (#1), Hyderabad, Gurugram, Mumbai dominate | Candidates should target these cities for max opportunity |
| 3 | Remote ≈ 40% of postings | Substantial remote opportunity in Indian tech market |
| 4 | Full-time = ~93% of employment-typed postings | Permanent roles drive the market |
| 5 | Mid-Senior level = ~82% of seniority-tagged postings | Entry-level roles are scarce; career switchers face challenges |
| 6 | IT Services & Consulting = ~48% of industry | Findings should not be generalised outside tech |
| 7 | EPAM Anywhere = ~23% of all postings | Single-company outlier skews aggregate stats |
| 8 | Lead Java & Senior Automation top titles | Java backend and QA/test-automation are in sustained demand |
| 9 | Median applicants = 23; max = 200 (capped) | Right-skewed; most postings attract modest interest |
| 10 | Posting age = strongest predictor of applicant count | Applying early is the single most actionable job-seeker strategy |

---
<a id='9'></a>
## 9. Machine Learning Workflow

**Business question:** *Can we predict, when a job posting goes live, whether it will
attract relatively high applicant interest — using only information publicly visible
at that moment?*

A reliable classifier allows:
- **Recruiters** to prioritise faster screening for high-competition roles
- **Job seekers** to anticipate competition levels before applying

**Algorithm:** Random Forest Classifier (200 trees, stratified 80/20 split)

---
<a id='10'></a>
## 10. Target Definition & Leakage Prevention

**Target variable:** `High_Applicant_Demand` (binary)

**Strict leakage rules:**
- `no_of_application` is the source of the target — **never** a predictor
- `High_Applicant_Demand` is the target itself — **never** used as input
- `job_ID` (raw identifier) is excluded
- **Median threshold is computed from the training set only** (test set remains unseen)
- Stratified 80/20 split ensures equal class proportions in train and test

In [ ]:
# ── Prepare ML dataset ────────────────────────────────────────
# Work only on rows that have a recorded applicant count
ml_df = df[df['no_of_application'].notna()].copy().reset_index(drop=True)

print(f'Rows with applicant data (ML-eligible): {len(ml_df):,}')
print(f'Rows excluded (no applicant count)    : {len(df) - len(ml_df):,}')

In [ ]:
# ── Step 1: Provisional split to derive training-set median ──
# We split BEFORE defining the target to prevent data leakage.
prov_train_idx, _ = train_test_split(
    ml_df.index, test_size=0.2, random_state=RANDOM_STATE, shuffle=True
)
median_threshold = ml_df.loc[prov_train_idx, 'no_of_application'].median()

print(f'Training-set median (threshold): {median_threshold:.0f} applicants')
print('Target: High_Applicant_Demand = 1 if applicants >= threshold, else 0')

In [ ]:
# ── Step 2: Define target using training-set threshold only ──
ml_df['High_Applicant_Demand'] = (
    ml_df['no_of_application'] >= median_threshold
).astype(int)

n_high = int(ml_df['High_Applicant_Demand'].sum())
n_low  = int((ml_df['High_Applicant_Demand'] == 0).sum())
print(f'Class distribution:')
print(f'  High demand (1): {n_high:,} ({n_high/len(ml_df)*100:.1f}%)')
print(f'  Low  demand (0): {n_low:,} ({n_low/len(ml_df)*100:.1f}%)')

In [ ]:
# ── Visualise target class balance ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
axes[0].bar(['Low Demand (0)', 'High Demand (1)'], [n_low, n_high],
            color=['#f59e0b', '#3b82d4'], edgecolor='white', width=0.5)
for i, val in enumerate([n_low, n_high]):
    axes[0].text(i, val + 10, str(val), ha='center', fontsize=10, fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].set_title('Target Class Distribution', fontweight='bold')

# Applicant count distributions by class
for cls, color, label in [(0, '#f59e0b', 'Low (0)'), (1, '#3b82d4', 'High (1)')]:
    data = ml_df[ml_df['High_Applicant_Demand'] == cls]['no_of_application']
    axes[1].hist(data, bins=30, alpha=0.65, color=color, label=label, edgecolor='white')
axes[1].axvline(median_threshold, color='#ef4444', linestyle='--', linewidth=2,
                label=f'Threshold = {median_threshold:.0f}')
axes[1].set_xlabel('Number of Applicants')
axes[1].set_ylabel('Postings')
axes[1].set_title('Applicant Count by Class', fontweight='bold')
axes[1].legend()

plt.suptitle(f'Target: High_Applicant_Demand (threshold = {median_threshold:.0f} applicants)',
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Step 3: Feature engineering & final stratified split ─────
CAT_COLS     = ['work_type', 'employment_type', 'seniority_level', 'emp_size']
NUM_COLS     = ['is_remote', 'log_followers', 'hours_since_posted']
FEATURE_LIST = CAT_COLS + NUM_COLS

model_df = ml_df[FEATURE_LIST + ['High_Applicant_Demand']].copy()

# Encode categoricals
encoders = {}
for col in CAT_COLS:
    model_df[col] = model_df[col].fillna('Unknown')
    le = LabelEncoder()
    model_df[col] = le.fit_transform(model_df[col].astype(str))
    encoders[col] = le

# Impute numeric NaNs with median
for col in NUM_COLS:
    model_df[col] = model_df[col].fillna(model_df[col].median())

X = model_df[FEATURE_LIST]
y = model_df['High_Applicant_Demand']

train_idx, test_idx = train_test_split(
    model_df.index,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)
X_train, X_test = X.loc[train_idx], X.loc[test_idx]
y_train, y_test = y.loc[train_idx], y.loc[test_idx]

print(f'Training set : {len(X_train):,} rows  (class balance: {y_train.mean():.3f})')
print(f'Test set     : {len(X_test):,} rows  (class balance: {y_test.mean():.3f})')
print(f'\nFeatures used: {FEATURE_LIST}')
print('\nLeakage check:')
print('  no_of_application in features:', 'no_of_application' in FEATURE_LIST)
print('  High_Applicant_Demand in features:', 'High_Applicant_Demand' in FEATURE_LIST)
print('  job_ID in features:', 'job_ID' in FEATURE_LIST)

---
<a id='11'></a>
## 11. Model Training & Evaluation

In [ ]:
# ── Train Random Forest ───────────────────────────────────────
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print('Model trained successfully.')
print(f'  n_estimators : {rf.n_estimators}')
print(f'  n_features   : {rf.n_features_in_}')

In [ ]:
# ── Confusion matrix ──────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Low (0)', 'High (1)']
)
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — Test Set', fontweight='bold')
plt.tight_layout()
plt.show()

print(f'TN={tn}  FP={fp}  FN={fn}  TP={tp}  |  Test set: {len(y_test):,} postings')

In [ ]:
# ── Feature importance ────────────────────────────────────────
feat_imp = (
    pd.DataFrame({'Feature': FEATURE_LIST, 'Importance': rf.feature_importances_})
    .sort_values('Importance', ascending=False)
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(feat_imp['Feature'][::-1], feat_imp['Importance'][::-1],
               color='#7c5cd8', edgecolor='white')
for bar, val in zip(bars, feat_imp['Importance'][::-1]):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', fontsize=9)
ax.set_xlabel('Feature Importance (Mean Decrease in Impurity)')
ax.set_title('Random Forest Feature Importances', fontweight='bold')
plt.tight_layout()
plt.show()

print('Feature Importances:')
print(feat_imp.to_string(index=False))

---
<a id='12'></a>
## 12. Accuracy, Precision, Recall & ROC-AUC

In [ ]:
# ── Performance metrics ───────────────────────────────────────
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec  = recall_score(y_test, y_pred, zero_division=0)
auc  = roc_auc_score(y_test, y_prob)

print('='*50)
print('   MODEL PERFORMANCE SUMMARY')
print('='*50)
print(f'   Model          : Random Forest (200 trees)')
print(f'   Train / Test   : 80% / 20% stratified')
print(f'   Train rows     : {len(X_train):,}')
print(f'   Test rows      : {len(X_test):,}')
print(f'   Threshold      : {median_threshold:.0f} applicants (training-set median)')
print('-'*50)
print(f'   Accuracy       : {acc:.4f}  ({acc*100:.2f}%)')
print(f'   Precision      : {prec:.4f}')
print(f'   Recall         : {rec:.4f}')
print(f'   ROC-AUC        : {auc:.4f}')
print('='*50)

In [ ]:
# ── ROC Curve ─────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(y_test, y_prob)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr, tpr, color='#3b82d4', linewidth=2.5,
        label=f'Random Forest  (AUC = {auc:.3f})')
ax.plot([0, 1], [0, 1], linestyle='--', color='#aaa', linewidth=1.5,
        label='No-skill baseline')
ax.fill_between(fpr, tpr, alpha=0.08, color='#3b82d4')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — High Applicant Demand Classifier', fontweight='bold')
ax.legend(loc='lower right')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Precision@k — analytical scenario (top 10%) ──────────────
# Score all ML-eligible rows
score_df = ml_df[FEATURE_LIST + ['no_of_application', 'High_Applicant_Demand']].copy()
for col in CAT_COLS:
    score_df[col] = score_df[col].fillna('Unknown')
    score_df[col] = encoders[col].transform(
        score_df[col]
        .where(score_df[col].isin(encoders[col].classes_), 'Unknown')
        .astype(str)
    )
for col in NUM_COLS:
    score_df[col] = score_df[col].fillna(score_df[col].median())

ml_df['Predicted_Prob'] = rf.predict_proba(score_df[FEATURE_LIST])[:, 1]

n_top10  = int(len(ml_df) * 0.10)
top10_df = ml_df.nlargest(n_top10, 'Predicted_Prob').reset_index(drop=True)
p_at_10  = top10_df['High_Applicant_Demand'].sum() / len(top10_df)

print(f'Analytical Scenario — Top 10% by Predicted Probability')
print(f'  Postings flagged       : {len(top10_df):,}')
print(f'  Precision @ top 10%    : {p_at_10:.1%}')
print(f'  (% of flagged that are truly high-demand)')
print()
show_cols = ['job', 'company_name', 'location', 'work_type', 'no_of_application', 'Predicted_Prob']
show_cols = [c for c in show_cols if c in top10_df.columns]
print('Top 10 highest-probability postings:')
print(top10_df[show_cols].head(10).to_string(index=False))

In [ ]:
# ── Metrics summary bar chart ─────────────────────────────────
metrics_dict = {'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'ROC-AUC': auc}

fig, ax = plt.subplots(figsize=(7, 4))
bar_colors = ['#3b82d4', '#7c5cd8', '#10b981', '#f59e0b']
bars = ax.bar(metrics_dict.keys(), metrics_dict.values(),
              color=bar_colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, metrics_dict.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', fontsize=10, fontweight='bold')
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('Model Performance Metrics (Test Set)', fontweight='bold')
ax.axhline(0.5, color='#aaa', linestyle='--', linewidth=1, label='No-skill baseline (0.5)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Model limitations ─────────────────────────────────────────
print('MODEL LIMITATIONS')
print('-'*60)
print('1. Binary threshold = training-set median (dataset-specific, not universal).')
print('2. LinkedIn caps displayed applicants at 200; right-tail distribution is censored.')
print(f'3. ~50% of postings ({len(df)-len(ml_df):,}) have no applicant count => excluded from ML.')
print('4. Features with high missingness (seniority_level: 38.8%, emp_size: 3.2%)')
print('   are imputed as "Unknown" — may dilute predictive signal.')
print('5. Model predicts applicant volume, NOT hiring outcomes.')
print('6. Dataset is India-centric; performance on other geographies is not validated.')

---
<a id='13'></a>
## 13. Final Business Insights & Recommendations

### For Job Seekers

| Recommendation | Evidence |
|---|---|
| **Target high-volume city hubs** | Bengaluru (905), Hyderabad (431), Gurugram (379), Mumbai (322) hold the majority of postings |
| **Prioritise in-demand technical skills** | Lead Java Software Engineer (158) and Senior Automation Tester (142) are the top titles; Java, .NET, Python, test-automation are in sustained demand |
| **Apply early** | Posting age is the #1 predictor of applicant volume (importance=0.37); applying within hours of posting going live reduces competition |
| **Consider smaller employers** | 1,001–5,000 employee companies post the most roles but also attract more applicants; 11–200 employee firms offer lower per-posting competition |
| **Leverage the remote market** | ~40% of postings are tagged Remote — meaningful access for candidates not tied to a specific city |

---

### For Recruiters / HR Teams

| Recommendation | Evidence |
|---|---|
| **Complete all posting fields** | 38.8% of postings omit seniority level; 37.2% omit industry — incomplete fields reduce match quality |
| **Invest in the company LinkedIn page** | log_followers is the #2 predictor of applicant volume (importance=0.29); broader reach = more applicants |
| **Publish high-priority roles early** | Applicant counts grow with posting age — publishing when target audience is active improves early application quality |
| **Use predicted demand scores for screening triage** | Postings ranked High Demand benefit from faster screening cycles; Low Demand postings may need description improvements |
| **Monitor competitor posting velocity** | EPAM Anywhere alone drives ~23% of postings — tracking competitor activity provides early signals of talent-supply shifts |

---

### For Career Analysts

| Recommendation | Evidence |
|---|---|
| **Control for EPAM Anywhere** | Single company = 23% of all postings; always report figures inclusive and exclusive of this outlier |
| **Control for posting age in applicant comparisons** | Posting age is the #1 driver; comparisons across work type / seniority must control for this variable |
| **Note the technology sector skew** | IT Services & Consulting = ~48% of industry-tagged postings; findings should not be generalised to non-tech markets |
| **Enrich with compensation data** | No salary fields exist — the most significant analytical gap; salary enrichment would enable demand-elasticity analysis |
| **Classify remote roles by geographic scope** | Many Remote postings list 'India' as location, implying domestic scope; a flag for truly location-agnostic roles would add value |

In [ ]:
# ── Final project summary ─────────────────────────────────────
print('LINKEDIN JOB MARKET ANALYSIS — PROJECT SUMMARY')
print('='*60)
print(f'Dataset          : linkdin_Job_data_cleaned.csv')
print(f'Rows             : {len(df):,}  |  Columns: {len(df.columns)}')
print(f'Companies        : {df["company_name"].nunique():,}')
print(f'Unique job titles: {df["job"].nunique():,}')
print(f'Unique locations : {df["location"].nunique():,}')
print()
print('Key EDA Metrics:')
print(f'  Top location     : {df["location"].value_counts().index[0]}')
print(f'  Top job title    : {df["job"].value_counts().index[0]} ({df["job"].value_counts().iloc[0]:,} postings)')
print(f'  Remote postings  : {df["is_remote"].sum():,} ({df["is_remote"].mean()*100:.1f}%)')
print(f'  Full-time share  : {(df["employment_type"]=="Full-time").sum():,} (~93%)')
print()
print('ML Model (Random Forest):')
print(f'  Target threshold : {median_threshold:.0f} applicants (training-set median)')
print(f'  Train / Test     : {len(X_train):,} / {len(X_test):,}')
print(f'  Accuracy         : {acc:.4f}')
print(f'  Precision        : {prec:.4f}')
print(f'  Recall           : {rec:.4f}')
print(f'  ROC-AUC          : {auc:.4f}')
print(f'  Top predictor    : {feat_imp.iloc[0]["Feature"]} (importance={feat_imp.iloc[0]["Importance"]:.3f})')
print(f'  2nd predictor    : {feat_imp.iloc[1]["Feature"]} (importance={feat_imp.iloc[1]["Importance"]:.3f})')
print('='*60)
print('Notebook complete.')